In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

PATH = "/grid_mnt/data__data.polcms/cms/adufour/DY_2026/analysis/combine/histograms.root"
# PATH = "histograms.root"  # if running locally after scp

f = uproot.open(PATH)

## What's inside?

In [ ]:
keys = f.keys()
print(f"{len(keys)} histograms found:\n")
for k in sorted(keys):
    h = f[k]
    print(f"  {k:<45s}  shape={h.values().shape}  integral={h.values().sum():.3f}")

## All processes (unrolled bins)

In [ ]:
CHANNEL = "triple_DY"

# collect nominal histograms (no Up/Down suffix)
nominal_keys = sorted([
    k for k in keys
    if k.startswith(f"{CHANNEL}/")
    and not k.endswith("Up") and not k.endswith("Down")
    and ";" not in k  # skip ROOT cycle suffixes if any
])

print("Nominal processes:", [k.split("/")[1] for k in nominal_keys])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = list(mcolors.TABLEAU_COLORS.values())

for i, key in enumerate(nominal_keys):
    h = f[key]
    vals = h.values().flatten()
    label = key.split("/")[1]
    ax.step(range(len(vals)), vals, where="post", label=label, color=colors[i % len(colors)])

ax.set_xlabel("Unrolled bin index")
ax.set_ylabel("Events")
ax.set_title(f"All nominal processes — {CHANNEL}")
ax.legend(fontsize=8, ncol=2)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## SM + EFT ratio

In [ ]:
sm_key   = f"{CHANNEL}/sm"
sm_vals  = f[sm_key].values().flatten()

# pick one operator to inspect
op = "cHDD"

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, proc_key in zip(axes, [f"{CHANNEL}/quad_{op}", f"{CHANNEL}/sm_lin_quad_{op}"]):
    proc_vals = f[proc_key].values().flatten()
    label     = proc_key.split("/")[1]
    
    ax.step(range(len(sm_vals)),   sm_vals,   where="post", label="sm",   color="steelblue")
    ax.step(range(len(proc_vals)), proc_vals, where="post", label=label,  color="tomato")
    ax.set_yscale("log")
    ax.set_xlabel("Unrolled bin")
    ax.set_ylabel("Events")
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

## Systematics: Up/Down envelopes for SM

In [ ]:
systs = ["qcd_scale", "pdf"]

fig, axes = plt.subplots(1, len(systs), figsize=(14, 4))

for ax, syst in zip(axes, systs):
    sm_nom = f[f"{CHANNEL}/sm"].values().flatten()
    sm_up  = f[f"{CHANNEL}/sm_{syst}Up"].values().flatten()
    sm_dn  = f[f"{CHANNEL}/sm_{syst}Down"].values().flatten()
    x = np.arange(len(sm_nom))
    
    ax.step(x, sm_nom, where="post", label="nominal", color="steelblue", lw=1.5)
    ax.step(x, sm_up,  where="post", label="up",      color="tomato",    lw=1, ls="--")
    ax.step(x, sm_dn,  where="post", label="down",    color="seagreen",  lw=1, ls="--")
    ax.fill_between(x, sm_dn, sm_up, step="post", alpha=0.15, color="gray")
    
    ax.set_title(f"SM — {syst}")
    ax.set_xlabel("Unrolled bin")
    ax.set_ylabel("Events")
    ax.set_yscale("log")
    ax.legend()

plt.tight_layout()
plt.show()

## Relative systematic uncertainty per bin

In [ ]:
fig, axes = plt.subplots(1, len(systs), figsize=(14, 4))

for ax, syst in zip(axes, systs):
    nom = f[f"{CHANNEL}/sm"].values().flatten()
    up  = f[f"{CHANNEL}/sm_{syst}Up"].values().flatten()
    dn  = f[f"{CHANNEL}/sm_{syst}Down"].values().flatten()
    mask = nom > 0
    
    rel_up = np.where(mask, (up - nom) / nom * 100, 0)
    rel_dn = np.where(mask, (nom - dn) / nom * 100, 0)
    x = np.arange(len(nom))
    
    ax.step(x,  rel_up, where="post", label="+σ", color="tomato")
    ax.step(x, -rel_dn, where="post", label="-σ", color="seagreen")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_title(f"SM relative uncertainty — {syst} (%)")
    ax.set_xlabel("Unrolled bin")
    ax.set_ylabel("Rel. unc. (%)")
    ax.legend()

plt.tight_layout()
plt.show()